In [1]:
import os

# Configuration des dépendances Kafka
os.environ["HOME"] = "/tmp"

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7 "
    "--conf spark.jars.ivy=/tmp/.ivy2 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession

# Configuration Spark pour l'exercice 5
spark = (
    SparkSession.builder
    .appName("Kafka_Weather_Aggregation")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)

print("Spark fonctionne !")
print("Version :", spark.version)
print("Application :", spark.sparkContext.appName)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /tmp/.ivy2/cache
The jars for the packages stored in: /tmp/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5cb35e9a-28a7-48d8-b601-2221c892e027;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1127ms :: artifacts dl 50ms
	:: modules in use

Spark fonctionne !
Version : 3.5.7
Application : Kafka_Weather_Aggregation


In [2]:
df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9094")
    .option("subscribe", "weather_transformed")
    .option("startingOffsets", "earliest")
    .load()
)

print("Lecture du topic weather_transformed configurée !")
df_kafka.printSchema()

Lecture du topic weather_transformed configurée !
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [3]:
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

schema = StructType([
    StructField("event_time", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("temperature", DoubleType()),
    StructField("windspeed", DoubleType()),
    StructField("weather_code", IntegerType()),
    StructField("wind_alert_level", StringType()),
    StructField("heat_alert_level", StringType())
])

df_weather = (
    df_kafka
    .select(
        from_json(col("value").cast("string"), schema).alias("data")
    )
    .select("data.*")
    .withColumn(
        "event_time",
        to_timestamp(col("event_time"))
    )
)

print("Données météo préparées !")
df_weather.printSchema()

Données météo préparées !
root
 |-- event_time: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- wind_alert_level: string (nullable = true)
 |-- heat_alert_level: string (nullable = true)



In [4]:
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

# Lecture du topic Kafka
df_kafka = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9094")
    .option("subscribe", "weather_transformed")
    .option("startingOffsets", "earliest")
    .load()
)

# Schéma des messages transformés
schema = StructType([
    StructField("event_time", StringType()),
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("temperature", DoubleType()),
    StructField("windspeed", DoubleType()),
    StructField("weather_code", IntegerType()),
    StructField("wind_alert_level", StringType()),
    StructField("heat_alert_level", StringType())
])

# Création de df_weather
df_weather = (
    df_kafka
    .select(
        from_json(
            col("value").cast("string"),
            schema
        ).alias("data")
    )
    .select("data.*")
    .withColumn(
        "event_time",
        to_timestamp(col("event_time"))
    )
)

print("df_weather créé avec succès !")
df_weather.printSchema()

df_weather créé avec succès !
root
 |-- event_time: timestamp (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- wind_alert_level: string (nullable = true)
 |-- heat_alert_level: string (nullable = true)



In [5]:
from pyspark.sql.functions import (
    window, sum as spark_sum,
    avg, min as spark_min,
    max as spark_max,
    when
)

df_aggregated = (
    df_weather
    .withWatermark("event_time", "1 minute")
    .groupBy(
        window("event_time", "1 minute", "30 seconds"),
        col("country"),
        col("city")
    )
    .agg(
        spark_sum(
            when(col("wind_alert_level") == "level_1", 1).otherwise(0)
        ).alias("wind_level_1"),

        spark_sum(
            when(col("wind_alert_level") == "level_2", 1).otherwise(0)
        ).alias("wind_level_2"),

        spark_sum(
            when(col("heat_alert_level") == "level_1", 1).otherwise(0)
        ).alias("heat_level_1"),

        spark_sum(
            when(col("heat_alert_level") == "level_2", 1).otherwise(0)
        ).alias("heat_level_2"),

        avg("temperature").alias("temperature_avg"),
        spark_min("temperature").alias("temperature_min"),
        spark_max("temperature").alias("temperature_max")
    )
    .withColumn(
        "total_alerts",
        col("wind_level_1")
        + col("wind_level_2")
        + col("heat_level_1")
        + col("heat_level_2")
    )
)

print("Agrégations configurées !")
df_aggregated.printSchema()

Agrégations configurées !
root
 |-- window: struct (nullable = true)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- wind_level_1: long (nullable = true)
 |-- wind_level_2: long (nullable = true)
 |-- heat_level_1: long (nullable = true)
 |-- heat_level_2: long (nullable = true)
 |-- temperature_avg: double (nullable = true)
 |-- temperature_min: double (nullable = true)
 |-- temperature_max: double (nullable = true)
 |-- total_alerts: long (nullable = true)



In [6]:
query_aggregation = (
    df_aggregated.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .trigger(processingTime="30 seconds")
    .start()
)

print("Agrégation météo démarrée !")
print("Streaming actif :", query_aggregation.isActive)

16:28:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c801b92b-87ee-44f5-a8a8-af3f3f229929. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
16:28:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Agrégation météo démarrée !
Streaming actif : True


16:28:03 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [7]:
import time

time.sleep(35)

print("Streaming actif :", query_aggregation.isActive)
print("Dernière progression :", query_aggregation.lastProgress)

if query_aggregation.exception() is not None:
    print("Erreur :", query_aggregation.exception())

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+--------------------+----------+------------+------------+------------+------------+---------------+---------------+---------------+------------+
|window                                    |country             |city      |wind_level_1|wind_level_2|heat_level_1|heat_level_2|temperature_avg|temperature_min|temperature_max|total_alerts|
+------------------------------------------+--------------------+----------+------------+------------+------------+------------+---------------+---------------+---------------+------------+
|{2026-11-10 17:02:30, 2026-11-10 17:03:30}|New Zealand         |Wellington|0           |0           |0           |0           |15.5           |15.5           |15.5           |0           |
|{2026-11-10 17:05:30, 2026-11-10 17:06:30}|France              |Paris     |0           |0           |0           |0           |12.7           

16:28:45 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 30000 milliseconds, but spent 42909 milliseconds


Streaming actif : True
Dernière progression : {'id': 'c4a7bbff-b0c3-45d0-a65a-a7663a1cf34f', 'runId': '2057fa13-c347-40e1-9869-10d51a8b4791', 'name': None, 'timestamp': '2026-11-10T16:28:02.483Z', 'batchId': 0, 'numInputRows': 129, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 3.009939801203976, 'durationMs': {'addBatch': 38516, 'commitOffsets': 75, 'getBatch': 43, 'latestOffset': 2110, 'queryPlanning': 1977, 'triggerExecution': 42856, 'walCommit': 85}, 'eventTime': {'avg': '2026-11-10T16:09:40.535Z', 'max': '2026-11-10T16:17:16.494Z', 'min': '2026-11-10T16:02:24.004Z', 'watermark': '1970-01-01T00:00:00.000Z'}, 'stateOperators': [{'operatorName': 'stateStoreSave', 'numRowsTotal': 154, 'numRowsUpdated': 154, 'allUpdatesTimeMs': 4197, 'numRowsRemoved': 0, 'allRemovalsTimeMs': 165, 'commitTimeMs': 8312, 'memoryUsedBytes': 102368, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 200, 'numStateStoreInstances': 200, 'customMetrics': {'loadedMapCacheHitCount': 0, 'loadedMapCache

-------------------------------------------
Batch: 1
-------------------------------------------
+------+-------+----+------------+------------+------------+------------+---------------+---------------+---------------+------------+
|window|country|city|wind_level_1|wind_level_2|heat_level_1|heat_level_2|temperature_avg|temperature_min|temperature_max|total_alerts|
+------+-------+----+------------+------------+------------+------------+---------------+---------------+---------------+------------+
+------+-------+----+------------+------------+------------+------------+---------------+---------------+---------------+------------+



In [8]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

print(
    "Nombre de partitions :",
    spark.conf.get("spark.sql.shuffle.partitions")
)

Nombre de partitions : 4


In [9]:
# Arrêter l'ancien streaming
if query_aggregation.isActive:
    query_aggregation.stop()

# Relancer avec 4 partitions
query_aggregation = (
    df_aggregated.writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", "false")
    .option("checkpointLocation", "/tmp/kafka_aggregation_checkpoint")
    .trigger(processingTime="30 seconds")
    .start()
)

print("Agrégation redémarrée !")
print("Streaming actif :", query_aggregation.isActive)

Agrégation redémarrée !
Streaming actif : True


16:30:08 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
16:30:08 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+--------------------+----------+------------+------------+------------+------------+---------------+---------------+---------------+------------+
|window                                    |country             |city      |wind_level_1|wind_level_2|heat_level_1|heat_level_2|temperature_avg|temperature_min|temperature_max|total_alerts|
+------------------------------------------+--------------------+----------+------------+------------+------------+------------+---------------+---------------+---------------+------------+
|{2026-11-10 17:02:00, 2026-11-10 17:03:00}|France              |Paris     |0           |0           |0           |0           |12.7           |12.7           |12.7           |0           |
|{2026-11-10 17:02:00, 2026-11-10 17:03:00}|New Zealand         |Wellington|0           |0           |0           |0           |15.5           

In [10]:
import time

time.sleep(35)

print("Streaming actif :", query_aggregation.isActive)
print("Dernière progression :", query_aggregation.lastProgress)

if query_aggregation.exception() is not None:
    print("Erreur :", query_aggregation.exception())

-------------------------------------------
Batch: 1
-------------------------------------------
+------+-------+----+------------+------------+------------+------------+---------------+---------------+---------------+------------+
|window|country|city|wind_level_1|wind_level_2|heat_level_1|heat_level_2|temperature_avg|temperature_min|temperature_max|total_alerts|
+------+-------+----+------------+------------+------------+------------+---------------+---------------+---------------+------------+
+------+-------+----+------------+------------+------------+------------+---------------+---------------+---------------+------------+

Streaming actif : True
Dernière progression : {'id': '4a02e648-9b2a-4cbc-8be2-190c2f2ee611', 'runId': '5ce1b35c-6178-46fc-a7dd-655f543cd629', 'name': None, 'timestamp': '2026-11-10T16:31:00.000Z', 'batchId': 2, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 26, 'triggerExecution': 29}, 'eventTime': {

In [11]:
print("Streaming actif :", query_aggregation.isActive)

print("Dernier batch :", query_aggregation.lastProgress["batchId"]
      if query_aggregation.lastProgress else "Aucun")

print("Messages traités :", query_aggregation.lastProgress["numInputRows"]
      if query_aggregation.lastProgress else 0)

Streaming actif : True
Dernier batch : 2
Messages traités : 0
